# Style2Fit — Step 3: Fine-tune SDXL
LoRA fine-tuning of Stable Diffusion XL on Fashion-Gen.
Targets semantic layers (down_blocks.2 + mid_block + up_blocks.0-1) to teach outfit coherence.

**Runtime:** A100 GPU. Takes ~45-60 min for 2000 steps.

In [ ]:
!pip install diffusers accelerate transformers datasets peft torch torchvision -q

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from pathlib import Path
from datasets import load_dataset
from diffusers import (
    AutoencoderKL, DDPMScheduler,
    StableDiffusionXLPipeline, UNet2DConditionModel,
)
from diffusers.optimization import get_scheduler
from peft import LoraConfig, get_peft_model
from transformers import CLIPTextModel, CLIPTextModelWithProjection, CLIPTokenizer

BASE_MODEL = 'stabilityai/stable-diffusion-xl-base-1.0'
RESOLUTION = 512
TRAIN_STEPS = 2000
BATCH_SIZE = 2
LR = 1e-4
LORA_RANK = 16

# Layers that encode semantic understanding (outfit coherence)
SEMANTIC_LAYERS = ['down_blocks.2', 'mid_block', 'up_blocks.0', 'up_blocks.1']

def is_semantic(name):
    return any(m in name for m in SEMANTIC_LAYERS)

print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Load SDXL components
print('Loading SDXL components (this takes a few minutes)...')
noise_scheduler = DDPMScheduler.from_pretrained(BASE_MODEL, subfolder='scheduler')
tokenizer_1 = CLIPTokenizer.from_pretrained(BASE_MODEL, subfolder='tokenizer')
tokenizer_2 = CLIPTokenizer.from_pretrained(BASE_MODEL, subfolder='tokenizer_2')
text_encoder_1 = CLIPTextModel.from_pretrained(BASE_MODEL, subfolder='text_encoder', torch_dtype=torch.bfloat16).cuda()
text_encoder_2 = CLIPTextModelWithProjection.from_pretrained(BASE_MODEL, subfolder='text_encoder_2', torch_dtype=torch.bfloat16).cuda()
vae = AutoencoderKL.from_pretrained(BASE_MODEL, subfolder='vae', torch_dtype=torch.bfloat16).cuda()
unet = UNet2DConditionModel.from_pretrained(BASE_MODEL, subfolder='unet')

# Freeze everything
for m in [vae, text_encoder_1, text_encoder_2, unet]:
    m.requires_grad_(False)

print('All components loaded.')

In [ ]:
# Apply LoRA to attention layers in semantic blocks only
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_RANK * 2,
    target_modules=['to_q', 'to_k', 'to_v', 'to_out.0'],
    lora_dropout=0.05,
    bias='none',
)
unet = get_peft_model(unet, lora_config)

# Freeze LoRA params outside semantic layers
for name, param in unet.named_parameters():
    if param.requires_grad and not is_semantic(name):
        param.requires_grad = False

# Cast entire unet to bfloat16 so all layers match
unet = unet.to(torch.bfloat16).cuda()

trainable = sum(p.numel() for p in unet.parameters() if p.requires_grad)
print(f'Trainable parameters: {trainable:,} ({trainable/1e6:.1f}M)')

In [ ]:
# Load fashion200k
print('Loading fashion200k dataset...')
ds = load_dataset('Marqo/fashion200k', split='data')
print(f'Dataset size: {len(ds)}')
print(f'Fields: {ds.column_names}')

image_transform = transforms.Compose([
    transforms.Resize(RESOLUTION, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.CenterCrop(RESOLUTION),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])

def collate_fn(examples):
    images, ids_1, ids_2 = [], [], []
    for ex in examples:
        try:
            img = ex['image'].convert('RGB')
        except Exception:
            return None
        images.append(image_transform(img))
        # fashion200k uses 'text' field
        caption = ex.get('text') or ex.get('description') or ex.get('caption') or ''
        t1 = tokenizer_1(caption, padding='max_length', max_length=77, truncation=True, return_tensors='pt')
        t2 = tokenizer_2(caption, padding='max_length', max_length=77, truncation=True, return_tensors='pt')
        ids_1.append(t1.input_ids[0])
        ids_2.append(t2.input_ids[0])
    return {
        'pixel_values': torch.stack(images),
        'input_ids_1': torch.stack(ids_1),
        'input_ids_2': torch.stack(ids_2),
    }

dataloader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True,
                        collate_fn=collate_fn, num_workers=2)
print('DataLoader ready.')

In [ ]:
from tqdm.notebook import tqdm

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, unet.parameters()),
    lr=LR, weight_decay=1e-2,
)
lr_scheduler = get_scheduler(
    'cosine', optimizer=optimizer,
    num_warmup_steps=100, num_training_steps=TRAIN_STEPS,
)

global_step = 0
unet.train()
Path('sdxl_checkpoints').mkdir(exist_ok=True)
progress = tqdm(total=TRAIN_STEPS, desc='Training SDXL', unit='step')

print(f'Training for {TRAIN_STEPS} steps...')
while global_step < TRAIN_STEPS:
    for batch in dataloader:
        if global_step >= TRAIN_STEPS:
            break
        if batch is None:
            continue

        with torch.no_grad():
            # cast pixel values to match VAE dtype (bfloat16)
            pixel_values = batch['pixel_values'].cuda().to(vae.dtype)
            latents = vae.encode(pixel_values).latent_dist.sample() * vae.config.scaling_factor

        noise = torch.randn_like(latents)
        timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps,
                                  (latents.shape[0],), device='cuda').long()
        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

        with torch.no_grad():
            enc1 = text_encoder_1(batch['input_ids_1'].cuda(), output_hidden_states=True)
            enc2 = text_encoder_2(batch['input_ids_2'].cuda(), output_hidden_states=True)
            prompt_embeds = torch.cat([enc1.hidden_states[-2], enc2.hidden_states[-2]], dim=-1)
            pooled_embeds = enc2[0]

        bs = latents.shape[0]
        add_time_ids = torch.tensor(
            [[RESOLUTION, RESOLUTION, 0, 0, RESOLUTION, RESOLUTION]] * bs,
            device='cuda', dtype=torch.bfloat16,
        )

        noise_pred = unet(
            noisy_latents, timesteps,
            encoder_hidden_states=prompt_embeds,
            added_cond_kwargs={'text_embeds': pooled_embeds, 'time_ids': add_time_ids},
        ).sample

        loss = F.mse_loss(noise_pred.float(), noise.float())
        loss.backward()
        torch.nn.utils.clip_grad_norm_(unet.parameters(), 1.0)
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

        global_step += 1
        progress.update(1)
        progress.set_postfix({'loss': f'{loss.item():.4f}', 'lr': f'{lr_scheduler.get_last_lr()[0]:.2e}'})

        if global_step % 500 == 0:
            unet.save_pretrained(f'sdxl_checkpoints/step-{global_step}')
            print(f'  Checkpoint saved at step {global_step}')

progress.close()
print('Training complete.')

In [ ]:
# Save final weights
unet.save_pretrained('sdxl_lora')
print('Saved to sdxl_lora/')

In [ ]:
# Quick before/after visual test
from diffusers import StableDiffusionXLPipeline
from peft import PeftModel
import matplotlib.pyplot as plt

test_prompt = (
    'Full body fashion editorial photo of a person wearing a cream knit sweater, '
    'straight-leg dark jeans, camel trench coat, white sneakers. '
    'Clean girl aesthetic. Full length shot, head to toe, shoes visible, '
    'soft natural lighting, clean background, professional fashion photography, sharp focus.'
)
negative_prompt = (
    'cropped, close up, portrait, headshot, cut off feet, cut off shoes, '
    'bad anatomy, blurry, low quality, cartoon'
)

# --- Before: base SDXL ---
pipe = StableDiffusionXLPipeline.from_pretrained(
    BASE_MODEL, torch_dtype=torch.bfloat16
).to('cuda')
pipe.enable_attention_slicing()
img_before = pipe(test_prompt, negative_prompt=negative_prompt,
                  num_inference_steps=30, height=1024, width=768).images[0]

# --- After: load LoRA via PEFT ---
pipe.unet = PeftModel.from_pretrained(pipe.unet, 'sdxl_lora')
pipe.unet = pipe.unet.to(torch.bfloat16)
img_after = pipe(test_prompt, negative_prompt=negative_prompt,
                 num_inference_steps=30, height=1024, width=768).images[0]

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(10, 14))
axes[0].imshow(img_before); axes[0].set_title('Before (base SDXL)', fontsize=14); axes[0].axis('off')
axes[1].imshow(img_after);  axes[1].set_title('After (fine-tuned)', fontsize=14);  axes[1].axis('off')
plt.tight_layout()
plt.savefig('before_after.png', dpi=150)
plt.show()
print('Saved before_after.png')

In [ ]:
!zip -r sdxl_lora.zip sdxl_lora/
from google.colab import files
files.download('sdxl_lora.zip')
files.download('before_after.png')